# ETL

In [3]:
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import numpy as np
import time
from datetime import datetime, date, timedelta
today = str(datetime.now().strftime("%Y%m%d"))
import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql import Row
import pandas as pd 
import glob
from pyspark.sql.functions import *
from deep_translator import GoogleTranslator
import country_converter as coco
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from geopy.geocoders import Nominatim
import logging
from pyspark.sql.window import Window

#region Configuration du logging
#Cette fonction sert à configurer le logger pour les différentes actions de l'ETL.
#Chaque fonction appelera le logger pour enregistrer les différentes étapes de l'ETL et les éventuelles erreurs qui pourraient survenir.
#Si le logger n'a pas été créé, la fonction le créera. Si le logger existe déjà, il sera utilisé tel quel et renvoyé.
def configure_logger(name=None,today=today):
    if name==None :
        name = f"etl_daily_{today}"
    log_dir = "logs"
    os.makedirs(log_dir, exist_ok=True) #Création du dossier de logs s'il n'existe pas
    logger = logging.getLogger(name)

    if not logger.handlers: #Évite de configurer plusieurs fois le logger si la fonction est appelée plusieurs fois
        logger.setLevel(logging.INFO) # DEBUG, INFO, WARNING, ERROR, CRITICAL
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)s | %(name)s | %(message)s" #Format : date | niveau de log (info/warning,error,etc...) | nom du logger (etl_daily) | message
        )

        #définition des handlers
        ##Les handlers permettent de définir où les logs sont envoyés
        stream_handler = logging.StreamHandler()
        stream_handler.setFormatter(formatter)
        file_handler = logging.FileHandler(
            f"{log_dir}/etl_{today}.log",
            mode='a', #mode 'a' pour append, 'w' pour écraser le fichier à chaque exécution
            encoding='utf-8')
        file_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
        logger.addHandler(stream_handler)

        logger.propagate = False  #Empêche les logs de remonter à la racine et d'être dupliqués si d'autres loggers sont utilisés dans le projet
    return logger
#endregion

#region Fonction de comparaison et d'ajout des nouvelles lignes (utilisée pour le load_data)
def compare_and_append_new_rows(spark_df, csv_path, logger=None):
    # Compare les nouvelles données avec les anciennes et retourne seulement les nouvelles lignes
    
    # Convertis Spark DF to Pandas
    new_data = spark_df.toPandas()
    
    if os.path.exists(csv_path):
        # Charge les données existantes
        existing_df = pd.read_csv(csv_path, sep=',')
        
        # Fait une jointure gauche pour garder les nouvelles lignes
        rows_to_append = (
            new_data
            .merge(
                existing_df,
                how='left',
                indicator=True #mets un indicateur technique sur la colonne avec les nouvelles lignes
            )
            .query('_merge == "left_only"')
            .drop(columns=['_merge']) #Se débarasse de la colonne technique temporaire
        )
        
        if logger:
            logger.info(f"Found {len(rows_to_append)} new rows to append to {csv_path}")
        
        return rows_to_append
    else:
        # File doesn't exist, return all data
        if logger:
            logger.info(f"Creating new file {csv_path}")
        
        return new_data
#endregion

spark = SparkSession.builder \
    .appName("BGES") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()
table_dir = "TABLES"
os.makedirs(table_dir, exist_ok=True)

#region Extraction des données
def extract_daily_data(logger=None,today=today) :
    if logger is None:
        logger = configure_logger(today=today)
    logger.info("Starting data extraction")
    #region Extraction des données
    try : 
        logger.info("Extracting personnel data")
        #Charger les fichiers de personnel
        fichiers_personnel = glob.glob('BDD_BGES/**/PERSONNEL_*.txt', recursive=True)
        psdf_personnel_list = [ps.read_csv(f, sep=';',index_col='ID_PERSONNEL') for f in fichiers_personnel]
        psdf_personnel = ps.concat(psdf_personnel_list, ignore_index=False)
    except Exception as e:
        logger.exception("Error during personnel data extraction")
        raise e

    try : 
        logger.info("Extracting informatic furnitures data")
        #Charger les fichiers de matériel informatique
        fichiers_mat_inf = glob.glob(f'BDD_BGES/**/MATERIEL_INFORMATIQUE_{today}.txt', recursive=True)
        psdf_mat_inf_list = [ps.read_csv(f, sep=';') for f in fichiers_mat_inf]
        if len(psdf_mat_inf_list) == 0:
            logger.warning("No informatic furnitures data found for today.")
            psdf_mat_inf = ps.DataFrame(columns=['ID_MATERIELINFO', 'TYPE', 'MODELE', 'ID_PERSONNEL', 'IMPACT'])
        else :
            psdf_mat_inf_temp = ps.concat(psdf_mat_inf_list, ignore_index=True)

            #Charge les données d'impact carbone du matériel informatique
            psdf_impact_matinf = ps.read_csv('BDD_BGES/materiel_informatique_impact.csv',sep=',')
            psdf_impact_matinf = psdf_impact_matinf.rename(columns={
                'Type': 'TYPE',
                'Impact': 'IMPACT',
                'Modèle': 'MODELE'
            })

            #Fusionne les données de matériel informatique avec les données d'impact carbone
            psdf_mat_inf = psdf_mat_inf_temp.merge(
                psdf_impact_matinf,
                on=['TYPE','MODELE'],
                how='inner'
            )
            #Impact est en kg/CO2, il faut le convertir en tonnes de CO2
            psdf_mat_inf['IMPACT'] = psdf_mat_inf['IMPACT'] / 1000
            psdf_mat_inf = psdf_mat_inf.set_index('ID_MATERIELINFO')

    except Exception as e:
        logger.exception("Error during informatic furnitures data extraction")
        raise e

    try :
        logger.info("Extracting mission data")
        #Charger les fichiers de mission
        fichiers_mission = glob.glob(f'BDD_BGES/**/MISSION_{today}.txt', recursive=True)
        psdf_mission_list = [ps.read_csv(f, sep=';',index_col='ID_MISSION') for f in fichiers_mission]
        if len(psdf_mission_list) == 0:
            logger.warning("No mission data found for today.")
            psdf_mission = ps.DataFrame(columns=['ID_MISSION', 'ID_PERSONNEL', 'VILLE_DEPART', 'PAYS_DEPART', 'VILLE_DESTINATION', 'PAYS_DESTINATION', 'MOYEN_TRANSPORT', 'DISTANCE_KM'])
        else :
            psdf_mission = ps.concat(psdf_mission_list, ignore_index=False)
    except Exception as e:
        logger.exception("Error during mission data extraction")
        raise e
    #endregion
    logger.info("Data extraction completed successfully")
    return psdf_personnel.to_spark(index_col='ID_PERSONNEL'), psdf_mat_inf.to_spark(index_col='ID_MATERIELINFO'), psdf_mission.to_spark(index_col='ID_MISSION')
#endregion

#region TRANSFORMATION DES DONNÉES
def transform_daily_data(logger=None,sdf_personnel=None, sdf_mat_inf=None, sdf_mission=None, today=today) :
    if logger is None:
        logger = configure_logger(today=today)

    #Mapping des rôles et des types de mission pour les uniformiser dans la base de données
    role_mapping = {
        #Economy
        'Ökonom': 'Economist',
        'Economiste': 'Economist',
        'Economist': 'Economist',

        #Engineer
        'Dateningenieur': 'Data Engineer',
        'Ingénieur Data': 'Data Engineer',
        'Data Engineer': 'Data Engineer',

        'Computeringenieur': 'Computer Engineer',
        'Ingénieur Informaticien': 'Computer Engineer',
        'Computer Engineer': 'Computer Engineer',

        #RH
        'Personalleiter': 'HR Manager',
        'DRH': 'HR Manager',
        'HRD': 'HR Manager',

        #Management
        'Führungskraft': 'Manager',
        'Leadership': 'Manager',
        'Cadre': 'Manager',  # choix le plus cohérent globalement
        'Business Executive': 'Manager'
    }
    mapping_list = [] 
    for k,v in role_mapping.items() :
        mapping_list+=[lit(k), lit(v)]
    mapping_role = create_map(*mapping_list)

    type_mission_mapping = {
        #Business
        'Geschäftstreffen': 'Business Meeting',
        'Business Meeting': 'Business Meeting',
        'Rencontre entreprises': 'Business Meeting',

        #Conference
        'Konferenz': 'Conference',
        'Conference': 'Conference',
        'Conférence': 'Conference',

        #Training
        'Schulung': 'Training',
        'Formation': 'Training',
        'Vocational Training': 'Training',

        #Meeting
        'Meeting': 'Internal Meeting',
        'Team Meeting': 'Internal Meeting',
        'Réunion': 'Internal Meeting',

        #Development
        'Entwicklung': 'Development',
        'Development': 'Development',
        'Développement': 'Development'
    }
    mapping_list = [] 
    for k,v in type_mission_mapping.items() :
        mapping_list+=[lit(k), lit(v)]
    mapping_type_mission = create_map(*mapping_list)    

    #region Création des tables de faits et de dimensions
    
    #Formatage des données dans le format des tables de la base de données
    #Table FAITS
    try : 
        logger.info("Creating table 'FAITS_MISSION'")
        FAITS_MISSION = sdf_mission.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select(
            'ID_MISSION',
            'ID_PERSONNEL',
            col('VILLE').alias('SITE'),
            year(to_date(col('DATE_MISSION'))).alias('ANNEE'),
            month(to_date(col('DATE_MISSION'))).alias('MOIS'),
            dayofmonth(to_date(col('DATE_MISSION'))).alias('JOUR')
        )
    except Exception as e:
        logger.exception(f"Error during mission facts table creation: {e}")
        raise e
    
    try : 
        logger.info("Creating table 'FAITS_MATERIEL_INFORMATIQUE'")
        FAITS_MATERIEL_INFORMATIQUE = sdf_mat_inf.join(sdf_personnel, on='ID_PERSONNEL', how='inner').select(
            'ID_PERSONNEL',
            'ID_MATERIELINFO',
            year(to_date(col('DATE_ACHAT'))).alias('ANNEE'),
            month(to_date(col('DATE_ACHAT'))).alias('MOIS'),
            dayofmonth(to_date(col('DATE_ACHAT'))).alias('JOUR'),
            col('VILLE').alias('SITE')
        )
    except Exception as e:
        logger.exception(f"Error during table 'FAITS_MATERIEL_INFORMATIQUE' creation: {e}")
        raise e

    #TABLE SITE
    try :
        logger.info("Creating table 'SITE'")
        SITE = sdf_personnel.select(col('VILLE').alias('SITE')).distinct()
    except Exception as e:
        logger.exception(f"Error during table 'SITE' creation: {e}")
        raise e
    
    #TABLE DATE
    try :
        logger.info("Creating table 'DATE'")
        DATE = sdf_mission.select(col('DATE_MISSION').alias('DATE')).join(sdf_mat_inf.select(col('DATE_ACHAT').alias('DATE')), on='DATE', how='outer')
        DATE = DATE.select(to_date(col('DATE'), 'yyyy-MM-dd').alias('DATE'))
        DATE = DATE.select(year(col('DATE')).alias('ANNEE'), month(col('DATE')).alias('MOIS'), dayofmonth(col('DATE')).alias('JOUR')).dropDuplicates()
    except Exception as e:
        logger.exception(f"Error during table 'DATE' creation: {e}")
        raise e

    #TABLE PERSONNEL
    try : 
        logger.info("Creating table 'PERSONNEL'")
        PERSONNEL = sdf_personnel.select('ID_PERSONNEL','PAYS','FONCTION_PERSONNEL',col('DT_NAISS').alias('DATE_NAISSANCE'))
        #uniformisation des rôles du personnel
        PERSONNEL = PERSONNEL.withColumn(
            "FONCTION_PERSONNEL",
            mapping_role[col("FONCTION_PERSONNEL")]
        )
    except Exception as e:
        logger.exception(f"Error during table 'PERSONNEL' creation: {e}")
        raise e
    
    #TABLE MATERIEL_INFORMATIQUE
    try : 
        logger.info("Creating table 'MATERIEL_INFORMATIQUE'")
        MATERIEL_INFORMATIQUE = sdf_mat_inf.select('ID_MATERIELINFO', 'TYPE', 'MODELE', 'IMPACT')
        MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
            "ECRAN",
            when(lower(col("TYPE")).contains("sans ecran"), "non")
            .otherwise("oui")
        ).withColumn(
            "TYPE_CLEAN",
            trim(split(lower(col("TYPE")), "sans ecran").getItem(0))
        ).select('ID_MATERIELINFO', col('TYPE_CLEAN').alias('TYPE'), 'MODELE', 'IMPACT', 'ECRAN')
    except Exception as e:
        logger.exception(f"Error during table 'MATERIEL_INFORMATIQUE' creation: {e}")
        raise e
    
    #TABLE LOCALISATION
    try : 
        logger.info("Creating table 'LOCALISATION'")
        cc = coco.CountryConverter()

        def get_continent(pays):
            try:
                continent = cc.convert(names=pays, to='continent')
                if continent == "not found":
                    return None
                return continent
            except:
                return None
            
        logger.info("Extracting unique cities and countries from data")
        LOCALISATION = sdf_mission.select(
            col("VILLE_DEPART").alias("VILLE"),
            initcap(col("PAYS_DEPART")).alias("PAYS")
        ).union(
            sdf_mission.select(
                col("VILLE_DESTINATION").alias("VILLE"),
                initcap(col("PAYS_DESTINATION")).alias("PAYS")
            )
        ).distinct()

        pays_uniques = [row["PAYS"] for row in LOCALISATION.select("PAYS").distinct().collect()]
        logger.info("Unique countries found: %s", pays_uniques)

        logger.info("Starting translation of unknown country names to English for continent mapping")
        translator = GoogleTranslator(source='auto', target='en')

        if os.path.exists('TABLES/CACHE_PAYS_CONTINENT.csv'):
            pays_csv = ps.read_csv('TABLES/CACHE_PAYS_CONTINENT.csv',sep=',')
            pays_map = set(pays_csv['PAYS'].to_list())
        else :
            pays_map = set()
        
        logger.info("Checking cache...")
        pays_a_traiter = set(pays_uniques) - pays_map
        logger.info("new countries to process: %s", pays_a_traiter)

        if len(pays_a_traiter) > 0 :
            logger.info("Translating and mapping continents for new countries: %s", pays_a_traiter)
            dict_pays_en = {
                pays: ("Sweden" if pays.lower() in ["suede", "suède"] else translator.translate(pays))
                for pays in pays_a_traiter
            }
            logger.info("Getting continent for each country")
            dict_pays_continent = {
                pays_fr: get_continent(dict_pays_en[pays_fr])
                for pays_fr in dict_pays_en
            }
            pd.DataFrame(dict_pays_continent.items(), columns=['PAYS', 'CONTINENT']).to_csv(
                'TABLES/CACHE_PAYS_CONTINENT.csv',
                mode='a',
                header=not os.path.exists('TABLES/CACHE_PAYS_CONTINENT.csv'),
                index=False
            )
        
        mapping_df = spark.read \
            .option("header", True) \
            .csv("TABLES/CACHE_PAYS_CONTINENT.csv")

        logger.info("Finalization of 'LOCALISATION' table with continent mapping")

        LOCALISATION = LOCALISATION.join(
            mapping_df,
            on="PAYS",
            how="left"
        )
    except Exception as e:
        logger.exception(f"Error during table 'LOCALISATION' creation.")
        raise e
    
    #region Table MISSION
    try :
        logger.info("Creating 'MISSION' table with geocoding and impact calculation")
        geolocator = Nominatim(user_agent="geo_app") #init du géolocaliseur

        #Récupérer toutes les villes uniques
        try :
            logger.info("Extracting unique cities and countries for geocoding")
            dep = sdf_mission.select(
                col("VILLE_DEPART").alias("VILLE"),
                col("PAYS_DEPART").alias("PAYS")
            )

            arr = sdf_mission.select(
                col("VILLE_DESTINATION").alias("VILLE"),
                col("PAYS_DESTINATION").alias("PAYS")
            )

            villes = dep.union(arr).distinct().toPandas()
        except Exception as e:
            logger.exception("Error during extraction of unique cities and countries")
            raise e

        #Géocoder les villes et stocker les résultats dans un cache local pour éviter de refaire les requêtes à chaque exécution
        coords = []
        if os.path.exists('TABLES/CACHE_COORD.csv'):
            coord_csv = ps.read_csv('TABLES/CACHE_COORD.csv',sep=',')
            villes_existantes = set(coord_csv['VILLE'].to_list())
        else :
            coord_csv = None
            villes_existantes = set()

        logger.info("Starting geocoding of cities")
        for _, row in villes.iterrows():
            ville = row["VILLE"]
            pays = row["PAYS"]

            if pd.notna(ville) and pd.notna(pays):
                if ville not in villes_existantes:
                    logger.info(f"Geocoding {ville}, {pays}")
                    query = f"{ville}, {pays}"
                    try:
                        loc = geolocator.geocode(query, timeout=10)
                        if loc:
                            coords.append((ville, pays, float(loc.latitude), float(loc.longitude)))
                        else:
                            logger.warning(f"Location not found for {query}")
                        time.sleep(1)
                    except Exception as e:
                        logger.warning(f"Error geocoding {query}: {e}")


        schema_coords = StructType([
            StructField("VILLE", StringType(), True),
            StructField("PAYS", StringType(), True),
            StructField("LAT", DoubleType(), True),
            StructField("LON", DoubleType(), True)
        ])

        if coords != []:
            coords_pdf = pd.DataFrame(coords, columns=["VILLE", "PAYS", "LAT", "LON"])
            coords_pdf.to_csv("TABLES/CACHE_COORD.csv", index=False, encoding="utf-8", mode='a',header=not os.path.exists('TABLES/CACHE_COORD.csv'))

        # Charger les coordonnées depuis le cache
        logger.info("Loading geocoded coordinates from cache")
        sdf_coords = spark.read \
            .option("header", True) \
            .option("inferSchema", True) \
            .csv("TABLES/CACHE_COORD.csv")

        sdf_coords = sdf_coords \
            .withColumn("LAT", col("LAT").cast("double")) \
            .withColumn("LON", col("LON").cast("double"))


        # Joindre les coordonnées départ / arrivée
        coords_dep = sdf_coords.select(
            col("VILLE").alias("VILLE_DEPART"),
            col("PAYS").alias("PAYS_DEPART"),
            col("LAT").alias("LAT_DEP"),
            col("LON").alias("LON_DEP")
        )

        coords_arr = sdf_coords.select(
            col("VILLE").alias("VILLE_DESTINATION"),
            col("PAYS").alias("PAYS_DESTINATION"),
            col("LAT").alias("LAT_ARR"),
            col("LON").alias("LON_ARR")
        )

        MISSION = sdf_mission \
            .join(coords_dep, on=["VILLE_DEPART", "PAYS_DEPART"], how="left") \
            .join(coords_arr, on=["VILLE_DESTINATION", "PAYS_DESTINATION"], how="left")

        # Calcul distance haversine en Spark
        try :
            logger.info("Calculating distances using Haversine formula")
            R = 6371.0 # Rayon de la Terre en km

            lat1 = radians(col("LAT_DEP"))
            lon1 = radians(col("LON_DEP"))
            lat2 = radians(col("LAT_ARR"))
            lon2 = radians(col("LON_ARR"))

            dlat = lat2 - lat1
            dlon = lon2 - lon1

            a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
            distance_expr = 2 * lit(R) * asin(sqrt(a))

            # Ajouter la distance au dataframe de mission
            MISSION = MISSION.withColumn(
                "DISTANCE_KM",
                when(
                    col("LAT_DEP").isNotNull() & col("LAT_ARR").isNotNull(),
                    distance_expr
                ).otherwise(
                    when(
                        col("VILLE_DEPART") == col("VILLE_DESTINATION"),
                        lit(4.0) #Selon une étude sur la distance des trajets urbains https://www.mdpi.com/2072-4292/13/9/1825
                    ).otherwise(None) #Si on n'a pas les coordonnées et que les villes sont différentes, on ne peut pas calculer la distance
                )
            )
        except Exception as e:
            logger.exception("Error during distance calculation.")
            raise e

        # Calcul impact carbone en Spark
        try :
            logger.info("Calculating carbon impact based on distance and transport mode")
            mode = lower(trim(col("TRANSPORT")))

            MISSION = MISSION.withColumn(
                "IMPACT_CARBONE",
                when(
                    col("DISTANCE_KM").isNotNull(),
                    when(mode == "avion",
                        when(col("DISTANCE_KM") < 1000, col("DISTANCE_KM") * 0.000258) # Coefficient pour les vols courts
                        .when(col("DISTANCE_KM") < 3500, col("DISTANCE_KM") * 0.000187) # Coefficient pour les vols moyens
                        .otherwise(col("DISTANCE_KM") * 0.000152) # Coefficient pour les vols longs
                    )
                    .when(mode == "train", col("DISTANCE_KM") * 0.000004) # Coefficient pour le train
                    .when(mode == "taxi", col("DISTANCE_KM") * 0.000192) # Coefficient pour le taxi
                    .when(mode == "transports en commun", col("DISTANCE_KM") * 0.00005) # Coefficient pour les transports en commun
                    .otherwise(None) # Si le mode de transport n'est pas reconnu, on ne peut pas calculer l'impact carbone
                ).otherwise(None)
            )
        except Exception as e:
            logger.exception("Error during carbon impact calculation.")
            raise e

        MISSION  = MISSION.select(
            "ID_MISSION",
            "TYPE_MISSION",
            "VILLE_DEPART",
            "VILLE_DESTINATION",
            "TRANSPORT",
            "ALLER_RETOUR",
            "IMPACT_CARBONE"
        )
        #uniformisation des types de mission
        MISSION = MISSION.withColumn(
            "TYPE_MISSION",
            mapping_type_mission[col("TYPE_MISSION")]
        )
    except Exception as e:
        logger.exception("Error during mission table enrichment with geocoding and impact calculation.")
        raise e
    #endregion
    
    # prédiction
    
    # 1) Nettoyage de base

    MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE \
        .withColumn("TYPE", lower(trim(col("TYPE")))) \
        .withColumn("MODELE", lower(trim(col("MODELE")))) \
        .withColumn(
            "TYPE",
            when((col("TYPE").isNull()) | (col("TYPE") == "") | (col("TYPE") == "nan"), None)
            .otherwise(col("TYPE"))
        ) \
        .withColumn(
            "MODELE",
            when((col("MODELE").isNull()) | (col("MODELE") == "") | (col("MODELE") == "nan"), None)
            .otherwise(col("MODELE"))
        )

    # 2) Prédiction TYPE par MODELE : type majoritaire pour chaque modèle
    modele_type_count = MATERIEL_INFORMATIQUE \
        .filter(col("TYPE").isNotNull() & col("MODELE").isNotNull()) \
        .groupBy("MODELE", "TYPE") \
        .agg(count("*").alias("nb"))

    window_modele = Window.partitionBy("MODELE").orderBy(desc("nb"))

    modele_type_pred = modele_type_count \
        .withColumn("rank", row_number().over(window_modele)) \
        .filter(col("rank") == 1) \
        .select("MODELE", col("TYPE").alias("TYPE_PRED_MODELE"))

    # 3) Fallback global : type le plus fréquent du dataset
    global_type_pred = MATERIEL_INFORMATIQUE \
        .filter(col("TYPE").isNotNull()) \
        .groupBy("TYPE") \
        .agg(count("*").alias("nb")) \
        .orderBy(desc("nb")) \
        .limit(1) \
        .collect()[0]["TYPE"]

    # 4) Imputation TYPE
    MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE \
        .join(modele_type_pred, on="MODELE", how="left") \
        .withColumn(
            "TYPE",
            coalesce(col("TYPE"), col("TYPE_PRED_MODELE"), lit(global_type_pred))
        ) \
        .drop("TYPE_PRED_MODELE")

    # 5) Détection écran AVANT nettoyage final
    MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
        "ECRAN",
        when(col("TYPE").contains("sans ecran"), "non")
        .when(col("TYPE").contains("sans écran"), "non")
        .when(col("TYPE").contains("avec ecran"), "oui")
        .when(col("TYPE").contains("avec écran"), "oui")
        .otherwise("non")
    )

    # 6) Nettoyage TYPE final
    MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
        "TYPE",
        regexp_replace(col("TYPE"), "sans ecran|sans écran", "")
    )

    MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.withColumn(
        "TYPE",
        trim(col("TYPE"))
    )

    # 7) Résultat final
    MATERIEL_INFORMATIQUE = MATERIEL_INFORMATIQUE.select(
        "ID_MATERIELINFO", "TYPE", "MODELE", "IMPACT", "ECRAN"
    )

        
    #endregion
    logger.info("Data transformation completed successfully")
    return FAITS_MISSION, FAITS_MATERIEL_INFORMATIQUE, SITE, DATE, PERSONNEL, MATERIEL_INFORMATIQUE, LOCALISATION, MISSION
#endregion

#region Chargement des données transformées dans des fichiers CSV
def load_daily_data(logger=None,daily_FAITS_MISSION=None, daily_FAITS_MATERIEL_INFORMATIQUE=None, daily_SITE=None, daily_DATE=None, daily_PERSONNEL=None, daily_MATERIEL_INFORMATIQUE=None, daily_LOCALISATION=None, daily_MISSION=None, today=today) :
    if logger is None:
        logger = configure_logger(today=today)
    logger.info("Starting data loading into CSV files")
    try :
        # Dictionary mapping CSV paths to Spark DataFrames
        tables_to_load = {
            'TABLES/FAITS_MISSION.csv': daily_FAITS_MISSION,
            'TABLES/FAITS_MATERIEL_INFORMATIQUE.csv': daily_FAITS_MATERIEL_INFORMATIQUE,
            'TABLES/MISSION.csv': daily_MISSION,
            'TABLES/SITE.csv': daily_SITE,
            'TABLES/DATE.csv': daily_DATE,
            'TABLES/PERSONNEL.csv': daily_PERSONNEL,
            'TABLES/MATERIEL_INFORMATIQUE.csv': daily_MATERIEL_INFORMATIQUE,
            'TABLES/LOCALISATION.csv': daily_LOCALISATION,
        }
        
        # Process each table
        for csv_path, spark_df in tables_to_load.items():
            # Drop duplicates for dimension tables
            if csv_path in ['TABLES/SITE.csv', 'TABLES/DATE.csv']:
                spark_df = spark_df.dropDuplicates()
            
            # Compare and get rows to append
            rows_to_append = compare_and_append_new_rows(spark_df, csv_path, logger)
            
            # Save to CSV
            if os.path.exists(csv_path):
                rows_to_append.to_csv(csv_path, mode='a', header=False, index=False)
            else:
                rows_to_append.to_csv(csv_path, header=True, index=False)
                
    except Exception as e:
        logger.exception("Error during data loading into CSV files")
        raise e
    finally:
        logger.info("Data loading completed successfully")
#endregion

# Simulation de l'ETL sur l'ensemble des données

In [2]:
date_min = date(2026, 4, 29)
date_max = date(2026, 11, 14)

current_date = date_min

while current_date <= date_max:
    sdf_personnel, sdf_mat_inf, sdf_mission = extract_daily_data(today=current_date.strftime("%Y%m%d"))
    daily_FAITS_MISSION, daily_FAITS_MATERIEL_INFORMATIQUE, daily_SITE, daily_DATE, daily_PERSONNEL, daily_MATERIEL_INFORMATIQUE, daily_LOCALISATION, daily_MISSION = transform_daily_data(sdf_personnel=sdf_personnel, sdf_mat_inf=sdf_mat_inf, sdf_mission=sdf_mission, today=current_date.strftime("%Y%m%d"))
    load_daily_data(daily_FAITS_MISSION=daily_FAITS_MISSION, daily_FAITS_MATERIEL_INFORMATIQUE=daily_FAITS_MATERIEL_INFORMATIQUE, daily_SITE=daily_SITE, daily_DATE=daily_DATE, daily_PERSONNEL=daily_PERSONNEL, daily_MATERIEL_INFORMATIQUE=daily_MATERIEL_INFORMATIQUE, daily_LOCALISATION=daily_LOCALISATION, daily_MISSION=daily_MISSION, today=current_date.strftime("%Y%m%d"))
    current_date += timedelta(days=1)


2026-05-10 13:26:48,008 | INFO | etl_daily | Starting data extraction
2026-05-10 13:26:48,009 | INFO | etl_daily | Extracting personnel data
2026-05-10 13:26:53,339 | INFO | etl_daily | Extracting informatic furnitures data
c:\Users\rzong\anaconda3\envs\nf26_env\Lib\site-packages\pyspark\pandas\utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
2026-05-10 13:26:54,957 | INFO | etl_daily | Extracting mission data
2026-05-10 13:26:56,102 | INFO | etl_daily | Data extraction completed successfully
2026-05-10 13:26:56,155 | INFO | etl_daily | Creating table 'FAITS_MISSION'
2026-05-10 13:26:56,193 | INFO | etl_daily | Creating table 'FAITS_MATERIEL_INFORMATIQUE'
2026-05-10 13:26:56,223 | INFO | etl_daily | Creating table 'SITE'
2026-05-10 13:26:56,235 | INFO | etl_daily | Creating table 'DATE'
2026-05-10 13:26:56,285 | INFO | e

KeyboardInterrupt: 